In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 16a: CNN Head — Hyperparameter Search
# Uses pre-extracted MobileNetV2 features from DDIM FashionMNIST
# Search on synth train/val features (no test set)
# Hyperparameter grid same as notebook 11a

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
import itertools
import gc

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

CLASSES    = 10
BATCH_SIZE = 128
EPOCHS     = 50

# HP grid — same as notebook 11a
PARAM_GRID = {
    'activation': ['relu', 'softmax'],
    'optimizer':  ['adadelta', 'adam', 'sgd'],
    'dropout':    [0.25, 0.5],
}

PROJECT_DIR  = '/content/drive/MyDrive/thesis_project'
FEATURES_DIR = os.path.join(PROJECT_DIR, 'data', 'extracted_features', 'seed_42', 'ddim')
RESULTS_DIR  = os.path.join(PROJECT_DIR, 'results', 'cnn_head', 'hp_search')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Load Pre-extracted Features ──────────────────────────────────────

F_train = np.load(os.path.join(FEATURES_DIR, 'fashion_mnist_x_train_features.npy'))
y_train = np.load(os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim',
                               'fashion_mnist', 'fashion_mnist_synthetic.npz'))['y_train']

F_val   = np.load(os.path.join(FEATURES_DIR, 'fashion_mnist_x_val_features.npy'))
y_val   = np.load(os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim',
                               'fashion_mnist', 'fashion_mnist_synthetic.npz'))['y_val']

print(f'F_train: {F_train.shape}, F_val: {F_val.shape}')

In [ ]:
# ── Cell 3: Hyperparameter Search ────────────────────────────────────────────

keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*[PARAM_GRID[k] for k in keys]))
print(f'Total runs: {len(combos)}\n')

records = []

for i, values in enumerate(combos):
    params   = dict(zip(keys, values))
    run_name = f"act{params['activation']}_opt{params['optimizer']}_do{params['dropout']}"

    tf.keras.backend.clear_session()
    gc.collect()

    head = models.Sequential([
        layers.Input(shape=(1280,)),
        layers.Dense(64, activation=params['activation']),
        layers.Dropout(params['dropout']),
        layers.Dense(CLASSES, activation='softmax'),
    ], name='cnn_head')

    head.compile(
        optimizer=params['optimizer'],
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = head.fit(
        F_train, y_train,
        validation_data=(F_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_val_acc    = max(history.history['val_accuracy'])
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc   = history.history['val_accuracy'][-1]

    records.append({
        'run_name':        run_name,
        'activation':      params['activation'],
        'optimizer':       params['optimizer'],
        'dropout':         params['dropout'],
        'best_val_acc':    round(best_val_acc, 4),
        'final_train_acc': round(final_train_acc, 4),
        'final_val_acc':   round(final_val_acc, 4),
    })

    print(f'[{i+1:02d}/{len(combos)}] {run_name:40s}  '
          f'best_val={best_val_acc:.4f}  train={final_train_acc:.4f}  val={final_val_acc:.4f}')

results_df = pd.DataFrame(records)

In [ ]:
# ── Cell 4: Results ───────────────────────────────────────────────────────────

results_ranked = results_df.sort_values('best_val_acc', ascending=False).reset_index(drop=True)
print('Search results ranked by best_val_acc:')
print(results_ranked.to_string(index=False))

best = results_ranked.iloc[0]
print(f'\nBest run: {best["run_name"]}')
print(f'  activation: {best["activation"]}')
print(f'  optimizer:  {best["optimizer"]}')
print(f'  dropout:    {best["dropout"]}')
print(f'  best_val_acc: {best["best_val_acc"]}')

results_df.to_csv(os.path.join(RESULTS_DIR, 'search_results.csv'), index=False)
results_ranked.to_csv(os.path.join(RESULTS_DIR, 'search_results_ranked.csv'), index=False)

In [ ]:
# ── Cell 5: Val Accuracy Plot ─────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))
run_names = results_ranked['run_name']
val_accs  = results_ranked['best_val_acc']
bars = ax.bar(range(len(run_names)), val_accs, color='steelblue', alpha=0.8)
bars[0].set_color('darkorange')
ax.set_xticks(range(len(run_names)))
ax.set_xticklabels(run_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Best Val Accuracy')
ax.set_title('Val Accuracy — All 12 CNN Head Search Runs (DDIM FashionMNIST Features)')
ax.set_ylim(0, 1)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'val_accuracy_all_runs.png'), dpi=150, bbox_inches='tight')
plt.close()
print(f'Results saved to: {RESULTS_DIR}')